# Rapport : plateforme d'analyse de logs

Ce rapport démontre le fonctionnement du pipeline : ingestion par générateurs, transformation fonctionnelle, analyse et moteur de règles de sécurité.

In [1]:
from logs_app.ingestion import lire_plusieurs_fichiers, parser_ligne
from logs_app.transformation import pipeline, router
from logs_app.analyse import compter_par, comparer_blacklist, ips_actives, detecter_force_brute
from logs_app.regles import apply_security_rules, regle_force_brute, regle_erreurs, regle_blacklist

lignes = lire_plusieurs_fichiers(["data/logs1.txt", "data/logs2.txt"])
logs = list(pipeline(lignes))
len(logs)

11

## 1. Ingestion et parsing
Les fichiers sont lus ligne par ligne avec `yield` et enchaînés avec `yield from` : le fichier n'est jamais chargé entièrement en mémoire. Chaque ligne est transformée en dictionnaire. Les lignes invalides renvoient `None`.

In [2]:
print(parser_ligne("2026-11-03 10:01:00 | ERROR | 172.16.0.3 | charlie | {\"code\": 500, \"service\": \"web\"}"))
print(parser_ligne("ligne cassee sans format"))

{'date': '2026-11-03 10:01:00', 'niveau': 'ERROR', 'ip': '172.16.0.3', 'utilisateur': 'charlie', 'message': {'code': 500, 'service': 'web'}, 'source': ('172.16.0.3', 'charlie')}
None


## 2. Transformation
Le pipeline enchaîne `map` (parsing, normalisation) et `filter` (`None` et `lambda` sur les niveaux valides). Le `match-case` reconnaît le format de chaque donnée.

In [3]:
for log in logs:
    print(log["resume"], "|", router(log["message"]))

[INFO] alice depuis 192.168.1.10 | Texte brut : Connexion reussie
[WARNING] bob depuis 10.0.0.5 | Texte brut : Mot de passe incorrect
[WARNING] bob depuis 10.0.0.5 | Texte brut : Mot de passe incorrect
[WARNING] bob depuis 10.0.0.5 | Texte brut : Mot de passe incorrect
[ERROR] bob depuis 10.0.0.5 | Texte brut : Compte bloque
[ERROR] charlie depuis 172.16.0.3 | Erreur applicative, code 500
[CRITICAL] admin depuis 45.33.12.8 | Erreur applicative, code 403
[INFO] alice depuis 192.168.1.10 | Texte brut : Deconnexion
[INFO] bob depuis 10.0.0.5 | Texte brut : Connexion reussie
[ERROR] eve depuis 8.8.4.4 | Texte brut : Acces refuse
[INFO] david depuis 192.168.1.20 | Texte brut : Connexion reussie


## 3. Analyse
Comptage avec `defaultdict`, comparaison avec la blacklist (`frozenset`) et détection de force brute.

In [4]:
print(compter_par(logs, "niveau"))
print(compter_par(logs, "utilisateur"))
print(comparer_blacklist(ips_actives(logs))["dangereuses_connectees"])
detecter_force_brute(logs)

{'INFO': 4, 'WARNING': 3, 'ERROR': 3, 'CRITICAL': 1}
{'alice': 2, 'bob': 5, 'charlie': 1, 'admin': 1, 'eve': 1, 'david': 1}
frozenset({'10.0.0.5', '45.33.12.8'})
OK : aucune attaque depuis 8.8.4.4
OK : aucune attaque depuis 45.33.12.8
ALERTE : force brute depuis 10.0.0.5
OK : aucune attaque depuis 172.16.0.3
OK : aucune attaque depuis 192.168.1.20
OK : aucune attaque depuis 192.168.1.10


{'10.0.0.5'}

## 4. Moteur de règles et audit
Les règles sont passées comme fonctions via `*args`, la configuration via `**kwargs`. Le moteur audite ses variables locales avec `locals()`.

In [5]:
apply_security_rules(regle_force_brute, regle_erreurs, regle_blacklist, logs=logs, seuil=3)

OK : aucune attaque depuis 8.8.4.4
OK : aucune attaque depuis 45.33.12.8
ALERTE : force brute depuis 10.0.0.5
OK : aucune attaque depuis 172.16.0.3
OK : aucune attaque depuis 192.168.1.20
OK : aucune attaque depuis 192.168.1.10


{'regle_force_brute': {'10.0.0.5'},
 'regle_erreurs': {'10.0.0.5', '172.16.0.3', '45.33.12.8', '8.8.4.4'},
 'regle_blacklist': frozenset({'10.0.0.5', '45.33.12.8'})}

## Conclusion
Le pipeline traite les logs sans saturer la mémoire, écarte les données invalides et détecte l'attaque par force brute depuis `10.0.0.5`, également présente dans la blacklist.